# 03 — Dynamic Programming for RL

## Learning Objectives
1. Implement iterative policy evaluation for a fixed policy
2. Build full policy iteration (evaluation + improvement) from scratch
3. Apply DP to GridWorld with obstacles and compare to value iteration
4. Explore modified and asynchronous DP variants for large state spaces

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from typing import Tuple, List, Dict

np.random.seed(42)

try:
    import torch
    torch.manual_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    device = 'cpu'

GRID = 4
N_S = GRID * GRID    # 16 states
N_A = 4              # up, down, left, right
GOAL = N_S - 1       # state 15
DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right

print(f'numpy {np.__version__}, torch={TORCH_AVAILABLE}, device={device}')
print(f'GridWorld: {GRID}x{GRID}, {N_S} states, {N_A} actions, goal=s{GOAL}')


## Level 1: Policy Evaluation — Iterative V_pi Computation

In [ ]:
def build_gridworld(grid_size=4, goal=None, obstacles=None, slip=0.0):
    """Build GridWorld MDP arrays P and R.

    Args:
        grid_size: Side length (n_states = grid_size^2).
        goal: Goal state index (default: last state).
        obstacles: Set of obstacle state indices with reward=-10.
        slip: Probability of random slide (perpendicular direction).

    Returns:
        P: (n_s, n_a, n_s) transition probabilities.
        R: (n_s, n_a) expected immediate rewards.
    """
    n = grid_size * grid_size
    if goal is None:
        goal = n - 1
    if obstacles is None:
        obstacles = set()
    P = np.zeros((n, 4, n))
    R = np.full((n, 4), -0.01)   # default step cost
    R[goal, :] = 0.0

    for s in range(n):
        r, c = divmod(s, grid_size)
        for a, (dr, dc) in enumerate(DELTAS):
            if slip > 0:
                perp = [(dc, -dr), (-dc, dr)]  # perpendicular directions
                outcomes = [(1 - slip, (dr, dc))] + [(slip / 2, d) for d in perp]
            else:
                outcomes = [(1.0, (dr, dc))]
            for prob, (d_r, d_c) in outcomes:
                nr = max(0, min(grid_size - 1, r + d_r))
                nc = max(0, min(grid_size - 1, c + d_c))
                ns = nr * grid_size + nc
                if s == goal or s in obstacles:
                    ns = s  # terminal states absorbing
                P[s, a, ns] += prob
                if ns == goal and s != goal:
                    R[s, a] = max(R[s, a], 1.0)
                elif ns in obstacles and s not in obstacles:
                    R[s, a] = min(R[s, a], -10.0)
    return P, R


def policy_evaluation(pi, P, R, gamma=0.9, theta=1e-5, max_iter=3000):
    """Iterative policy evaluation: compute V_pi for a fixed stochastic policy."""
    n_s = P.shape[0]
    V = np.zeros(n_s)
    deltas = []
    for it in range(max_iter):
        # Vectorized: V_new[s] = sum_a pi[s,a] * (R[s,a] + gamma * P[s,a,:] @ V)
        future = (P * V[None, None, :]).sum(axis=2)  # (n_s, n_a)
        V_new = (pi * (R + gamma * future)).sum(axis=1)
        delta = np.max(np.abs(V_new - V))
        deltas.append(delta)
        V = V_new
        if delta < theta:
            break
    return V, deltas


P, R = build_gridworld(grid_size=4)
pi_uniform = np.ones((N_S, N_A)) / N_A  # random policy
print('Policy evaluation with uniform random policy (gamma=0.9):')
V_pi, eval_deltas = policy_evaluation(pi_uniform, P, R, gamma=0.9)
print(f'  Converged in {len(eval_deltas)} iterations')
print(f'  V(s=0)={V_pi[0]:.4f}, V(s=14)={V_pi[14]:.4f}, V(s=15)={V_pi[15]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(eval_deltas, color='steelblue')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Max Delta')
axes[0].set_title('Policy Evaluation Convergence'); axes[0].grid(True, alpha=0.3)
im = axes[1].imshow(V_pi.reshape(4, 4), cmap='coolwarm', origin='upper')
for i in range(4):
    for j in range(4):
        s = i * 4 + j
        axes[1].text(j, i, f'{V_pi[s]:.2f}', ha='center', va='center', fontsize=8)
axes[1].set_title('V_pi (uniform random)'); plt.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.savefig('rl_03_policy_eval.png', dpi=80, bbox_inches='tight')
plt.show()


## Level 2: Full Policy Iteration

In [ ]:
def policy_improvement(V, P, R, gamma=0.9):
    """Greedy policy improvement: pi_new(s) = argmax_a [R(s,a) + gamma*P(s,a,:)@V]."""
    Q = R + gamma * (P @ V)       # (n_s, n_a)
    pi_new = np.zeros_like(R)     # will become deterministic one-hot
    best_actions = Q.argmax(axis=1)
    for s, a in enumerate(best_actions):
        pi_new[s, a] = 1.0
    return pi_new, Q


def policy_iteration(P, R, gamma=0.9, eval_theta=1e-5, max_pi_iter=200):
    """Full policy iteration: alternate eval and improvement until stable.

    Returns:
        V_star: Optimal value array.
        pi_star: Optimal deterministic policy (one-hot stochastic form).
        pi_history: List of policies at each iteration.
        eval_iters: Number of policy evaluation iterations per PI step.
    """
    n_s, n_a = R.shape
    pi = np.ones((n_s, n_a)) / n_a   # start with uniform random
    pi_history = [pi.copy()]
    eval_iters_list = []

    for pi_step in range(max_pi_iter):
        # Step 1: Policy Evaluation
        V, deltas = policy_evaluation(pi, P, R, gamma=gamma, theta=eval_theta)
        eval_iters_list.append(len(deltas))

        # Step 2: Policy Improvement
        pi_new, Q = policy_improvement(V, P, R, gamma=gamma)

        # Check stability: policy unchanged -> converged
        policy_stable = np.all(pi_new.argmax(axis=1) == pi.argmax(axis=1))
        pi = pi_new
        pi_history.append(pi.copy())

        if policy_stable:
            print(f'  Policy iteration converged in {pi_step+1} steps '
                  f'({sum(eval_iters_list)} total eval iterations)')
            return V, pi, pi_history, eval_iters_list, Q

    print(f'  Policy iteration did NOT converge in {max_pi_iter} steps')
    return V, pi, pi_history, eval_iters_list, Q


print('Policy iteration on 4x4 GridWorld (gamma=0.9):')
V_pi_star, pi_star, pi_hist, eval_it, Q_pi = policy_iteration(P, R, gamma=0.9)

print(f'  V*(s=0)={V_pi_star[0]:.4f}, V*(s=14)={V_pi_star[14]:.4f}')
print(f'  Policy iteration steps: {len(pi_hist)-1}')
print(f'  Eval iters per step: {eval_it}')

action_names = ['Up', 'Down', 'Left', 'Right']
arrow_map = {0: (0, -0.38), 1: (0, 0.38), 2: (-0.38, 0), 3: (0.38, 0)}

def plot_policy_arrows(ax, pi_det, V, title, grid=4, goal=15, obstacles=None):
    """Plot policy as arrows on a grid with V(s) annotations."""
    if obstacles is None:
        obstacles = set()
    for s in range(grid * grid):
        r, c = divmod(s, grid)
        if s == goal:
            ax.add_patch(plt.Rectangle((c-0.45, r-0.45), 0.9, 0.9, color='gold'))
            ax.text(c, r, 'G', ha='center', va='center', fontsize=12, fontweight='bold')
        elif s in obstacles:
            ax.add_patch(plt.Rectangle((c-0.45, r-0.45), 0.9, 0.9, color='#c0392b', alpha=0.7))
            ax.text(c, r, 'X', ha='center', va='center', fontsize=12, color='white')
        else:
            a_opt = pi_det[s]
            dx, dy = arrow_map[a_opt]
            ax.annotate('', xy=(c + dx, r - dy), xytext=(c, r),
                        arrowprops=dict(arrowstyle='->', color='navy', lw=1.8))
            ax.text(c + 0.3, r - 0.3, f'{V[s]:.2f}', fontsize=6, color='dimgray')
    ax.set_xlim(-0.6, grid - 0.4); ax.set_ylim(-0.6, grid - 0.4)
    ax.set_xticks(range(grid)); ax.set_yticks(range(grid))
    ax.set_aspect('equal'); ax.invert_yaxis()
    ax.set_title(title); ax.grid(True, alpha=0.3)

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))
pi_star_det = pi_star.argmax(axis=1)
plot_policy_arrows(axes2[0], pi_star_det, V_pi_star, 'Optimal Policy (policy iteration)')
# Compare to initial uniform random
pi_rand_det = np.random.randint(N_A, size=N_S)
V_rand, _ = policy_evaluation(np.eye(N_A)[pi_rand_det], P, R)
plot_policy_arrows(axes2[1], pi_rand_det, V_rand, 'Random Policy V')
plt.tight_layout(); plt.savefig('rl_03_policy_iter.png', dpi=80, bbox_inches='tight')
plt.show()


## Real-World Example 1: Policy Iteration on 8x8 GridWorld with Obstacles

In [ ]:
# 8x8 GridWorld with 3 obstacles (R=-10), goal=s63
GRID8 = 8
GOAL8 = GRID8 * GRID8 - 1   # state 63
OBSTACLES8 = {18, 27, 45}   # obstacle states with heavy penalty

print(f'Building 8x8 GridWorld ({GRID8*GRID8} states), obstacles={OBSTACLES8}')
P8, R8 = build_gridworld(grid_size=GRID8, goal=GOAL8, obstacles=OBSTACLES8)

print('Running policy iteration on 8x8 GridWorld...')
V8_star, pi8_star, pi8_hist, eval8_it, Q8_pi = policy_iteration(P8, R8, gamma=0.95)

pi8_det = pi8_star.argmax(axis=1)
print(f'  V*(start=s0): {V8_star[0]:.4f}')
print(f'  PI steps: {len(pi8_hist)-1}, total eval iters: {sum(eval8_it)}')

# Plot 8x8 value heatmap and policy arrows
fig_8, axes_8 = plt.subplots(1, 2, figsize=(14, 6))
im8 = axes_8[0].imshow(V8_star.reshape(GRID8, GRID8), cmap='Blues', origin='upper')
for i in range(GRID8):
    for j in range(GRID8):
        s = i * GRID8 + j
        if s == GOAL8:
            label = 'G'
        elif s in OBSTACLES8:
            label = 'X'
        else:
            label = f'{V8_star[s]:.1f}'
        axes_8[0].text(j, i, label, ha='center', va='center', fontsize=6)
axes_8[0].set_title('V*(s) — 8x8 with obstacles')
plt.colorbar(im8, ax=axes_8[0])

# Policy arrows (sampled for clarity at 8x8)
ax8a = axes_8[1]
small_arrow = {0: (0, -0.35), 1: (0, 0.35), 2: (-0.35, 0), 3: (0.35, 0)}
for s in range(GRID8 * GRID8):
    r, c = divmod(s, GRID8)
    if s == GOAL8:
        ax8a.add_patch(plt.Rectangle((c-0.45, r-0.45), 0.9, 0.9, color='gold'))
        ax8a.text(c, r, 'G', ha='center', va='center', fontsize=9, fontweight='bold')
    elif s in OBSTACLES8:
        ax8a.add_patch(plt.Rectangle((c-0.45, r-0.45), 0.9, 0.9, color='#c0392b', alpha=0.8))
        ax8a.text(c, r, 'X', ha='center', va='center', fontsize=9, color='white')
    else:
        a_opt = pi8_det[s]
        dx, dy = small_arrow[a_opt]
        ax8a.annotate('', xy=(c + dx, r - dy), xytext=(c, r),
                      arrowprops=dict(arrowstyle='->', color='navy', lw=1.2))
ax8a.set_xlim(-0.6, GRID8 - 0.4); ax8a.set_ylim(-0.6, GRID8 - 0.4)
ax8a.set_xticks(range(GRID8)); ax8a.set_yticks(range(GRID8))
ax8a.set_aspect('equal'); ax8a.invert_yaxis()
ax8a.set_title('Optimal Policy (pi*) — avoids obstacles')
ax8a.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_03_8x8_gridworld.png', dpi=80, bbox_inches='tight')
plt.show()

# Verify policy avoids obstacles
print('\nActions near obstacles (policy should route around):')
for obs in OBSTACLES8:
    for s_near in [obs - 1, obs + 1, obs - GRID8, obs + GRID8]:
        if 0 <= s_near < GRID8 * GRID8 and s_near not in OBSTACLES8 and s_near != GOAL8:
            a = pi8_det[s_near]
            ns_check = s_near  # just show action name
            print(f'  Near obs {obs}: s={s_near} takes action {action_names[a]}')


## Real-World Example 2: Modified Policy Iteration (Truncated Evaluation)

In [ ]:
# Modified PI: run only K eval iterations per improvement step (faster)
# K=1 -> value iteration; K=inf -> full policy iteration

def modified_policy_iteration(P, R, gamma=0.9, K=5, max_pi_iter=200, theta=1e-6):
    """Modified policy iteration with K-step truncated policy evaluation.

    Args:
        K: Max evaluation steps per improvement (1 = VI, infinity = full PI).

    Returns:
        V_star: Converged value function.
        total_updates: Total number of Bellman updates (eval steps * states).
        residuals: Max Bellman residual at each PI step.
    """
    n_s = R.shape[0]
    V = np.zeros(n_s)
    pi = np.ones((n_s, N_A)) / N_A
    total_updates = 0
    residuals = []

    for step in range(max_pi_iter):
        # Truncated evaluation: K Bellman backups
        for _ in range(K):
            future = (P * V[None, None, :]).sum(axis=2)
            V_new = (pi * (R + gamma * future)).sum(axis=1)
            total_updates += n_s
            V = V_new

        # Greedy improvement
        Q = R + gamma * (P @ V)
        pi_new = np.zeros_like(pi)
        pi_new[np.arange(n_s), Q.argmax(axis=1)] = 1.0

        residual = np.max(np.abs(Q.max(axis=1) - V))
        residuals.append(residual)

        if np.all(pi_new.argmax(axis=1) == pi.argmax(axis=1)) and residual < theta:
            print(f'  K={K}: converged in {step+1} PI steps, {total_updates} state-updates')
            return V, total_updates, residuals
        pi = pi_new

    return V, total_updates, residuals


# Compare K=1 (VI), K=5 (modified PI), K=50 (near-full PI)
K_values = [1, 5, 10, 50]
mpi_results = {}
for K in K_values:
    V_k, updates_k, res_k = modified_policy_iteration(P, R, gamma=0.9, K=K)
    mpi_results[K] = {'V': V_k, 'updates': updates_k, 'residuals': res_k}

fig_mpi, axes_mpi = plt.subplots(1, 2, figsize=(14, 5))
for K, color in zip(K_values, ['steelblue', 'darkorange', 'green', 'red']):
    res = mpi_results[K]['residuals']
    axes_mpi[0].semilogy(res, label=f'K={K}', color=color, linewidth=2)
axes_mpi[0].set_xlabel('PI Step'); axes_mpi[0].set_ylabel('Max Bellman Residual')
axes_mpi[0].set_title('Modified PI: Residual vs PI Steps'); axes_mpi[0].legend()
axes_mpi[0].grid(True, alpha=0.3)

Ks = list(K_values)
total_updates = [mpi_results[K]['updates'] for K in Ks]
axes_mpi[1].bar([str(K) for K in Ks], total_updates,
                color=['steelblue', 'darkorange', 'green', 'red'], alpha=0.8)
axes_mpi[1].set_xlabel('K (eval steps per PI iteration)')
axes_mpi[1].set_ylabel('Total State Updates')
axes_mpi[1].set_title('Total Computation (state updates) vs K')
axes_mpi[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('rl_03_modified_pi.png', dpi=80, bbox_inches='tight')
plt.show()

print(f'  {"K":>4} {"PI steps":>10} {"State updates":>15} {"V*(s=0)":>10}')
for K in K_values:
    n_steps = len(mpi_results[K]['residuals'])
    print(f'  {K:>4} {n_steps:>10} {mpi_results[K]["updates"]:>15} '
          f'{mpi_results[K]["V"][0]:>10.4f}')


## Real-World Example 3: Asynchronous DP with Prioritized State Selection

In [ ]:
# Asynchronous DP: update only high-residual states (prioritized sweeping)
# Critical for large state spaces where full sweeps are expensive

def async_dp_prioritized(P, R, gamma=0.9, tol=1e-6, max_updates=10000):
    """Prioritized asynchronous DP: always update state with largest Bellman error.

    Mimics Dyna-Q's background planning without a model-free wrapper.
    Returns V_star and update history for analysis.
    """
    n_s = R.shape[0]
    V = np.zeros(n_s)
    update_counts = np.zeros(n_s, dtype=int)
    residual_history = []
    v_history_s0 = []  # track s=0 value convergence

    for step in range(max_updates):
        # Compute Bellman errors for all states
        Q = R + gamma * (P @ V)         # (n_s, n_a)
        bellman_errors = np.abs(Q.max(axis=1) - V)
        max_err = bellman_errors.max()
        residual_history.append(max_err)
        v_history_s0.append(V[0])

        if max_err < tol:
            print(f'  Async prioritized DP: converged in {step} updates')
            break

        # Update the state with the highest Bellman error
        s_update = np.argmax(bellman_errors)
        V[s_update] = Q[s_update].max()
        update_counts[s_update] += 1

    return V, residual_history, update_counts, v_history_s0


print('Running async prioritized DP on 4x4 GridWorld...')
V_async, res_async_dp, upd_counts, v0_hist = async_dp_prioritized(P, R, gamma=0.9)

# Compare with sync VI
def sync_vi_simple(P, R, gamma=0.9, tol=1e-6, max_iter=2000):
    n_s = R.shape[0]
    V = np.zeros(n_s)
    residuals = []
    for _ in range(max_iter):
        Q = R + gamma * (P @ V)
        V_new = Q.max(axis=1)
        residuals.append(np.max(np.abs(V_new - V)))
        V = V_new
        if residuals[-1] < tol:
            break
    return V, residuals

V_sync_vi, res_sync_vi = sync_vi_simple(P, R, gamma=0.9)

fig_async, axes_async = plt.subplots(1, 3, figsize=(16, 5))

# Convergence comparison
axes_async[0].semilogy(res_async_dp, label='Async Prioritized', color='green', linewidth=2)
# Sync VI: 1 update/iter = 1 residual; compare apples-to-apples by #state updates
sync_updates = np.arange(1, len(res_sync_vi)+1) * N_S
axes_async[0].semilogy(sync_updates, res_sync_vi, label='Sync VI (x N_states)', color='steelblue', linewidth=2)
axes_async[0].set_xlabel('State Updates'); axes_async[0].set_ylabel('Max Bellman Residual')
axes_async[0].set_title('Convergence: Async Prioritized vs Sync VI')
axes_async[0].legend(); axes_async[0].grid(True, alpha=0.3)

# Update frequency heatmap
im_upd = axes_async[1].imshow(upd_counts.reshape(4, 4), cmap='Reds', origin='upper')
for i in range(4):
    for j in range(4):
        s = i * 4 + j
        axes_async[1].text(j, i, str(upd_counts[s]), ha='center', va='center', fontsize=9)
axes_async[1].set_title('Update Frequency per State')
plt.colorbar(im_upd, ax=axes_async[1])

# V(s=0) trajectory
axes_async[2].plot(v0_hist, color='darkorange', linewidth=2)
axes_async[2].axhline(V_sync_vi[0], color='navy', linestyle='--', label='V*(s=0) sync')
axes_async[2].set_xlabel('Update Steps'); axes_async[2].set_ylabel('V(s=0)')
axes_async[2].set_title('V(s=0) Convergence (Async Prioritized)')
axes_async[2].legend(); axes_async[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_03_async_dp.png', dpi=80, bbox_inches='tight')
plt.show()

print(f'Most updated state: s={upd_counts.argmax()} ({upd_counts.max()} times)')
print(f'Least updated: s={upd_counts.argmin()} ({upd_counts.min()} times)')
print('High-value states near goal get updated most; far states propagate slowly.')


## Comparison: Policy Iteration vs Value Iteration

In [ ]:
# Compare PI vs VI: speed, quality, computation

def value_iteration_full(P, R, gamma=0.9, tol=1e-6, max_iter=2000):
    """Full synchronous value iteration returning full history."""
    n_s = R.shape[0]
    V = np.zeros(n_s)
    residuals = []
    history = [V.copy()]
    for i in range(max_iter):
        Q = R + gamma * (P @ V)
        V_new = Q.max(axis=1)
        residuals.append(np.max(np.abs(V_new - V)))
        history.append(V_new.copy())
        V = V_new
        if residuals[-1] < tol:
            break
    return V, Q, residuals, history


print('Comparing PI vs VI on 4x4 GridWorld (gamma=0.9):')
V_vi, Q_vi, res_vi, hist_vi = value_iteration_full(P, R, gamma=0.9)
V_pi_cmp, pi_cmp, _, eval_it_cmp, Q_pi_cmp = policy_iteration(P, R, gamma=0.9)

# Total Bellman updates: VI = len(res_vi)*N_S, PI = sum(eval_it)*N_S
vi_updates = len(res_vi) * N_S
pi_updates = sum(eval_it_cmp) * N_S

print(f'  VI : {len(res_vi)} iterations, {vi_updates} state-updates, final residual {res_vi[-1]:.2e}')
print(f'  PI : {len(eval_it_cmp)} PI steps, {pi_updates} state-updates (eval), residual={res_vi[-1]:.2e}')
print(f'  V* agreement: max|V_VI - V_PI| = {np.max(np.abs(V_vi - V_pi_cmp)):.2e}')

fig_cmp, axes_cmp = plt.subplots(1, 2, figsize=(13, 5))

# Convergence curves
axes_cmp[0].semilogy(res_vi, label=f'Value Iteration ({len(res_vi)} iters)', color='steelblue', linewidth=2)
# PI total eval residual (last delta of each eval phase)
gamma_cmp = 0.9  # discount factor for this comparison
V_pi_trace = np.zeros(N_S)
pi_trace = np.ones((N_S, N_A)) / N_A
pi_res_trace = []
for step_t in range(20):  # trace PI steps
    V_pi_trace, deltas_t = policy_evaluation(pi_trace, P, R)
    Q_t = R + gamma_cmp * (P @ V_pi_trace)
    pi_res_trace.append(deltas_t[-1])
    pi_new_t = np.zeros_like(pi_trace)
    pi_new_t[np.arange(N_S), Q_t.argmax(axis=1)] = 1.0
    if np.all(pi_new_t.argmax(axis=1) == pi_trace.argmax(axis=1)):
        break
    pi_trace = pi_new_t
axes_cmp[0].semilogy(pi_res_trace, label=f'Policy Iteration ({len(pi_res_trace)} PI steps)', color='darkorange', linewidth=2, linestyle='--')
axes_cmp[0].set_xlabel('Iteration/Step'); axes_cmp[0].set_ylabel('Residual')
axes_cmp[0].set_title('VI vs PI Convergence'); axes_cmp[0].legend(); axes_cmp[0].grid(True, alpha=0.3)

# Bar: total state updates
axes_cmp[1].bar(['Value Iteration', 'Policy Iteration'], [vi_updates, pi_updates],
                color=['steelblue', 'darkorange'], alpha=0.8, width=0.5)
axes_cmp[1].set_ylabel('Total State Updates (Bellman backups)')
axes_cmp[1].set_title('Computational Cost: VI vs PI')
axes_cmp[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('rl_03_vi_vs_pi.png', dpi=80, bbox_inches='tight')
plt.show()

print('\nPolicy Iteration advantage: far fewer iterations (quadratic convergence).')
print('Value Iteration advantage: simpler, no nested evaluation loop.')


## Key Takeaways

**Core idea:** Dynamic programming solves MDPs exactly when the model (P, R) is known. Policy iteration alternates between full policy evaluation and greedy improvement, converging in far fewer iterations than value iteration but at higher per-step cost.

| Method | Per-step cost | # Iterations | Best for |
|--------|--------------|--------------|----------|
| Value Iteration | 1 sweep | Many (linear) | Simple, easy to implement |
| Policy Iteration | Full eval + improvement | Few (quadratic) | Convergence speed |
| Modified PI (K) | K-step eval | Between VI and PI | Tunable trade-off |
| Async Prioritized | One state update | Variable | Large sparse MDPs |

**Failure modes:**
- Policy evaluation diverges: check gamma < 1 and proper absorbing terminal states
- Policy iteration cycles: numerical instability — ensure deterministic tie-breaking
- Slow convergence: try prioritized sweeping or increase K in modified PI

**Related:** [01-mdp](01-markov-decision-processes.ipynb), [02-bellman](02-bellman-equations.ipynb), [04-mc](04-monte-carlo-methods.ipynb)

## Exercises

1. **Stochastic policy:** Run policy iteration starting from a hand-crafted policy (e.g., always go Right). Compare convergence speed to starting from uniform random.
2. **Obstacle density:** Add 10 obstacles to the 8x8 grid. Does PI still converge? How many steps?
3. **Modified PI sweep:** Plot total state updates vs K for K in {1, 2, 5, 10, 50, 100}. What K minimizes total updates?
4. **Async update order:** Compare prioritized, random, and fixed (state 0 first) async DP. Which converges fastest?